# Optimal Execution: the Almgren-Chriss Frontier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/lectures/optimal_execution.ipynb)

You have a big position to unwind. Sell it all now and you pay huge **market impact**; sell it slowly
and you're exposed to price **risk** the whole time. Almgren-Chriss (2000) makes this precise: choose
the liquidation trajectory that minimises `E[cost] + λ · Var[cost]`, where `λ` is your risk aversion.
We trace the trade-off with [`finmlsim`](https://github.com/convexpi/finmlsim).

## Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q "finmlsim[analysis]"
import numpy as np
import matplotlib.pyplot as plt
import finmlsim as fms
print('ready — finmlsim', fms.__version__)

## 1. Patient vs. aggressive trajectories

Higher `λ` (more risk-averse) front-loads selling to cut exposure; `λ → 0` (risk-neutral) trades
evenly to minimise impact. Each curve is the position remaining over the trading day.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
for lam, c in [(1e-7, "steelblue"), (1e-6, "seagreen"), (1e-5, "crimson")]:
    res = fms.simulate.almgren_chriss(X=1.0, T=1.0, N=50, sigma=0.02, eta=2.5e-6, lam=lam)
    ax.plot(np.linspace(0, 1, len(res["x"])), res["x"], lw=1.8, label=f"λ={lam:g}")
ax.set_xlabel("time (fraction of horizon)"); ax.set_ylabel("shares remaining")
ax.set_title("Almgren-Chriss liquidation trajectories"); ax.legend(); ax.axhline(0, color="k", lw=0.6)
plt.tight_layout(); plt.show()

## 2. The efficient frontier

Sweep `λ` and plot each strategy's **expected cost** against its **cost risk** (standard deviation).
The frontier is the set of un-improvable trade-offs — you can't cut risk without paying more impact.

In [ ]:
lams = np.logspace(-8, -4, 25)
pts = [fms.simulate.almgren_chriss(X=1.0, T=1.0, N=50, sigma=0.02, eta=2.5e-6, lam=l) for l in lams]
E = np.array([p["E_cost"] for p in pts]); S = np.array([p["sd_cost"] for p in pts])

fig, ax = plt.subplots(figsize=(7.5, 4))
sc = ax.scatter(S, E * 1e4, c=np.log10(lams), cmap="viridis", s=30)
ax.set_xlabel("cost risk  (sd of cost)"); ax.set_ylabel("expected cost (bps)")
ax.set_title("Execution efficient frontier (colour = log₁₀ λ)")
plt.colorbar(sc, label="log₁₀ risk aversion λ"); plt.tight_layout(); plt.show()
print("risk-neutral (λ→0): lowest expected cost, highest risk;  risk-averse: the opposite.")

## Why this matters for ConvexPi

Mission 8 shows turnover-times-spread eating your alpha at the *portfolio* level. Almgren-Chriss is
the same trade-off one level down — inside a single order. A strategy's *paper* alpha is only real if
it survives both: the per-order impact of getting in, and the per-rebalance cost of staying in.

## Takeaways

1. **Impact vs. timing risk is the core execution trade-off** — trading faster cuts risk but costs impact.
2. **λ (risk aversion) picks your point on the frontier** — there is no single "optimal" speed, only a trade-off.
3. **Impact is convex in size** — which is why *capacity* (Mission 8) is finite.

→ Pair this with Mission 8 (the cost of trading) for the portfolio-level view.